In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

In [3]:
spark = SparkSession.builder \
    .appName("Recomendador_Anime_Final") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

In [4]:
path = 'gs://big-data-lunes-20260223/reviews/'

In [5]:
df_animes = spark.read.csv(f"{path}animes.csv", header=True, inferSchema=True)
df_profiles = spark.read.csv(f"{path}profiles.csv", header=True, inferSchema=True)

In [6]:
#reviews tienen reviews multilinea, modifiquemos
df_reviews = spark.read.csv(
    f"{path}reviews.csv", 
    header=True, 
    inferSchema=True, 
    multiLine=True, 
    escape='"', 
    quote='"'
).select("profile", "anime_uid", "score")

df_reviews = df_reviews.filter(
    col("score").cast("float").isNotNull() & 
    col("profile").isNotNull() & 
    col("anime_uid").isNotNull()
)

In [7]:
df_reviews.show()

+---------------+---------+-----+
|        profile|anime_uid|score|
+---------------+---------+-----+
| DesolatePsyche|    34096|    8|
|      baekbeans|    34599|   10|
|           skrn|    28891|    7|
|   edgewalker00|     2904|    9|
|aManOfCulture99|     4181|   10|
|          eneri|     2904|   10|
| Waffle_Empress|    16664|    6|
|   NIGGER_BONER|     2904|    8|
|         jchang|     2904|   10|
|    shadowsplat|     4181|    4|
|   angelsreview|     4672|    8|
| CalebTheMenace|     4181|    9|
|        Kiethol|     4181|   10|
|          Eanki|    34599|    8|
|      NekoKyupi|    34599|    9|
|          12sed|     5114|   10|
|  OVERPOWERED99|    30276|    8|
|  MrAnimeCrunch|    30276|   10|
|    JoJo_Stalin|    30276|    7|
|        Kaishuu|     4181|   10|
+---------------+---------+-----+
only showing top 20 rows



In [8]:
#El algoritmo ALS solo acepta números enteros para los IDs.
indexer_user = StringIndexer(inputCol="profile", outputCol="user_id_num", handleInvalid="skip")
indexer_anime = StringIndexer(inputCol="anime_uid", outputCol="item_id_num", handleInvalid="skip")

In [9]:
%%time
pipeline = Pipeline(stages=[indexer_user, indexer_anime])
pipeline_model = pipeline.fit(df_reviews)
df_prep = pipeline_model.transform(df_reviews).cache() # CACHE: Estabiliza el Broadcast

26/02/28 07:56:15 WARN DAGScheduler: Broadcasting large task binary with size 1758.5 KiB
26/02/28 07:56:23 WARN DAGScheduler: Broadcasting large task binary with size 1758.3 KiB


CPU times: user 27.4 ms, sys: 3.61 ms, total: 31 ms
Wall time: 24.2 s


In [10]:
%%time
als = ALS(
    maxIter=10, 
    regParam=0.1, 
    userCol="user_id_num", 
    itemCol="item_id_num", 
    ratingCol="score",
    coldStartStrategy="drop",
    nonnegative=True
)

model = als.fit(df_prep)

26/02/28 07:56:31 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:41 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:41 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:43 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:45 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:46 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:49 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:50 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:53 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:55 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:57 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:56:59 WARN DAGScheduler: Broadcasting larg

CPU times: user 89.9 ms, sys: 23.3 ms, total: 113 ms
Wall time: 49.2 s


In [11]:
user_recs = model.recommendForAllUsers(5)

In [12]:
recs_flat = user_recs.withColumn("rec", explode("recommendations")) \
    .select("user_id_num", col("rec.item_id_num").alias("item_id_num"))

In [13]:
mapping_user = df_prep.select("user_id_num", "profile").distinct()
mapping_anime = df_prep.select("item_id_num", "anime_uid").distinct()

In [14]:
final_recs = recs_flat.join(mapping_user, "user_id_num") \
    .join(mapping_anime, "item_id_num") \
    .join(df_animes, col("anime_uid") == col("uid")) \
    .select(col("profile").alias("Usuario"), col("title").alias("Recomendacion"))

In [16]:
spark.sparkContext.setLogLevel("ERROR")

In [17]:
print("--- Listado de Recomendaciones Finales ---")
final_recs.show(10, truncate=False)

--- Listado de Recomendaciones Finales ---


26/02/28 07:58:55 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/02/28 07:58:55 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:58:55 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:58:56 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:58:57 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/02/28 07:59:04 WARN TaskSetManager: Lost task 22.0 in stage 238.0 (TID 414) (hive-learning-cluster-w-2.us-central1-a.c.big-data-lunes-2026-1.internal executor 11): java.lang.StackOverflowError
	at java.base/java.lang.Exception.<init>(Exception.java:102)
	at java.base/java.lang.ReflectiveOperationException.<init>(ReflectiveOperationException.java:89)
	at java.base/java.lang.reflect.InvocationTargetException.<init>(InvocationTargetException.java:73)
	at jdk.internal.reflect.GeneratedMethodAccessor5.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.Delegat

+------------+---------------+
|Usuario     |Recomendacion  |
+------------+---------------+
|dadnaya     |Akazukin Chacha|
|Jesswem     |Akazukin Chacha|
|Reflex96    |Akazukin Chacha|
|TRI_Mike    |Akazukin Chacha|
|BulkKarma   |Akazukin Chacha|
|rexguardian |Akazukin Chacha|
|PixelPenguin|Akazukin Chacha|
|vavoysh     |Akazukin Chacha|
|Room304     |Akazukin Chacha|
|Thedarkaxe  |Akazukin Chacha|
+------------+---------------+
only showing top 10 rows



In [19]:
spark.stop()

# Trucutru, usando checkpoints

In [11]:
import os
import warnings
import pandas as pd
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, explode
warnings.filterwarnings('ignore')

In [2]:
checkpoint_dir = "/user/checkpoint_anime"
!hdfs dfs -mkdir {checkpoint_dir}


mkdir: `/user/checkpoint_anime': File exists


In [3]:
# 2. CONFIGURACIÓN ROBUSTA DE SPARK
# - Aumentamos el Stack (-Xss4m) para evitar StackOverflow
# - Búfer de Kryo (512m) para evitar Buffer Overflow
# - Serialización Kryo para velocidad
spark = SparkSession.builder \
    .appName("Anime_Recommender_Final_Pro") \
    .config("spark.executor.extraJavaOptions", "-Xss4m") \
    .config("spark.driver.extraJavaOptions", "-Xss4m") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

26/02/28 08:23:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
spark.sparkContext.setLogLevel("ERROR")

In [5]:
spark.sparkContext.setCheckpointDir(checkpoint_dir)

26/02/28 08:23:27 WARN SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/user/checkpoint_anime' appears to be on the local filesystem.


In [6]:
# 3. CONFIGURACIÓN DE CHECKPOINT (Vital para ALS)
# Esto corta el linaje y evita que el Grafo de Spark sea demasiado profundo
checkpoint_path = "/user/checkpoint_anime_dir"
!hdfs dfs -mkdir {checkpoint_path}
spark.sparkContext.setCheckpointDir(checkpoint_path)

mkdir: `/user/checkpoint_anime_dir': File exists


26/02/28 08:23:35 WARN SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/user/checkpoint_anime_dir' appears to be on the local filesystem.


In [34]:
!hdfs dfs -ls /user

Found 14 items
drwxr-xr-x   - root hadoop          0 2026-02-28 08:05 /user/checkpoint_anime
drwxr-xr-x   - root hadoop          0 2026-02-28 08:05 /user/checkpoint_anime_dir
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/dataproc
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/hbase
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/hdfs
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/hive
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/mapred
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/pig
drwxr-xr-x   - root hadoop          0 2026-02-28 08:04 /user/root
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/solr
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/spark
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/yarn
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/zeppelin
drwxrwxrwt   - hdfs hadoop          0 2026-02-28 03:58 /user/zookeeper


In [7]:
path = 'gs://big-data-lunes-20260223/reviews/'

In [8]:
df_animes = spark.read.csv(f"{path}animes.csv", header=True, inferSchema=True)
df_profiles = spark.read.csv(f"{path}profiles.csv", header=True, inferSchema=True)

In [12]:
#reviews tienen reviews multilinea, modifiquemos
df_reviews = spark.read.csv(
    f"{path}reviews.csv", 
    header=True, 
    inferSchema=True, 
    multiLine=True, 
    escape='"', 
    quote='"'
).select("profile", "anime_uid", "score")

df_reviews = df_reviews.filter(
    col("score").cast("float").isNotNull() & 
    col("profile").isNotNull() & 
    col("anime_uid").isNotNull()
)

In [13]:
user_indexer = StringIndexer(inputCol="profile", outputCol="user_id").setHandleInvalid("skip")
anime_indexer = StringIndexer(inputCol="anime_uid", outputCol="item_id").setHandleInvalid("skip")

In [14]:
pipeline = Pipeline(stages=[user_indexer, anime_indexer])
prep_model = pipeline.fit(df_reviews)
df_final = prep_model.transform(df_reviews)

26/02/28 08:25:14 WARN DAGScheduler: Broadcasting large task binary with size 1758.5 KiB
26/02/28 08:25:21 WARN DAGScheduler: Broadcasting large task binary with size 1758.3 KiB


In [15]:
df_final = df_final.checkpoint()

26/02/28 08:25:24 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


In [16]:
(train, test) = df_final.randomSplit([0.8, 0.2], seed=42)

In [17]:
print(">>> Entrenando el modelo (Matrix Factorization)...")
als = ALS(
    maxIter=10, 
    regParam=0.1, 
    userCol="user_id", 
    itemCol="item_id", 
    ratingCol="score",
    coldStartStrategy="drop",
    checkpointInterval=2,
    nonnegative=True
)
model = als.fit(train)

>>> Entrenando el modelo (Matrix Factorization)...


In [18]:
# 5. EVALUACIÓN DEL ERROR
print(">>> Calculando métricas de error...")
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="score", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"RESULTADO: El RMSE del modelo es {rmse:.4f}")

>>> Calculando métricas de error...


RESULTADO: El RMSE del modelo es 1.8210


In [19]:
# 6. GENERACIÓN DE RECOMENDACIONES (Top 5)
print(">>> Generando recomendaciones personalizadas...")
user_recs = model.recommendForAllUsers(5)

# Aplanar y unir con títulos reales
final_recs = user_recs.withColumn("rec", F.explode("recommendations")) \
    .select("user_id", F.col("rec.item_id").alias("item_id"))

>>> Generando recomendaciones personalizadas...


In [21]:
# Mapeo de vuelta a nombres
mapping_user = df_final.select("user_id", "profile").distinct()
mapping_anime = df_final.select("item_id", "anime_uid").distinct()

recs_readable = final_recs.join(mapping_user, "user_id") \
    .join(mapping_anime, "item_id") \
    .join(df_animes, col("anime_uid") == col("uid")) \
    .select(
        col("profile").alias("Usuario"), 
        col("title").alias("Recomendacion"),
        col("genre").alias("Genero") # Aprovechamos que df_animes tiene el género
    )

In [22]:
# 7. ANÁLISIS DE ANIMES MÁS POPULARES EN LAS RECOMENDACIONES
print(">>> Analizando tendencias globales...")
popular_recs = recs_readable.groupBy("Recomendacion").count() \
    .orderBy(F.col("count").desc())

popular_recs.show(10, truncate=False)

>>> Analizando tendencias globales...


+--------------------------+-----+
|Recomendacion             |count|
+--------------------------+-----+
|Nee Summer!               |16176|
|Train Heroes              |13747|
|Shakugan no Shana Specials|12328|
|Shinseiki Inma Seiden     |8758 |
|Mouryou no Nie            |7300 |
|Papa to Odorou            |6160 |
|Ryuudouji Shimon no Inbou |5824 |
|Tiger Mask                |5803 |
|Tokyo Futago Athletic     |4637 |
|Orangutan                 |4127 |
+--------------------------+-----+
only showing top 10 rows



In [ ]:
# 8. VISUALIZACIÓN DE RESULTADOS
import plotly.express as px

# 1. Convertimos los top 15 a Pandas
top_pd = popular_recs.limit(15).toPandas()

# 2. Creamos la gráfica
fig = px.bar(
    top_pd, 
    x='count', 
    y='Recomendacion', 
    orientation='h', 
    title='<b>Top 15 Animes más Recomendados por el Sistema</b>',
    labels={'count': 'Número de Usuarios', 'Recomendacion': 'Título del Anime'},
    color='count', 
    color_continuous_scale='Viridis', # Un gradiente profesional de azul a amarillo
    template='plotly_dark'
)

# 3. Ajustes estéticos adicionales
fig.update_layout(
    yaxis={'categoryorder':'total ascending'}, # Asegura que el más grande esté arriba
    title_x=0.5, # Centra el título
    coloraxis_showscale=False # Oculta la barra de color lateral para mayor limpieza
)

# 4. Mostrar
fig.show()

In [ ]:
!git status